## Kaggle Setup & Install Dependencies

In [2]:
import subprocess, sys

packages = [
    'codecarbon==2.3.4',
    # 'fasttext-wheel',
    # 'transformers==4.36.2',
    # 'datasets==2.16.1',
    # 'sentencepiece',
    # 'accelerate',
    # 'memory-profiler',
    # 'psutil',
    # 'scipy',
    'seaborn',
]

for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)

print('✅ All packages installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.6/181.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 2.6 MB/s eta 0:00:00
✅ All packages installed.


In [3]:
# ── Global Imports & Config ───────────────────────────────────────────────────
import os, json, time, glob, warnings, tracemalloc, tempfile
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import combinations
from sklearn.metrics import accuracy_score, f1_score, classification_report
warnings.filterwarnings('ignore')

# ── Reproducibility Seeds ─────────────────────────────────────────────────────
SEEDS   = [42, 123, 456]          # 3 seeds → enables paired t-tests
DOMAINS = ['restaurant', 'laptop']
LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

# ── Directory Setup ───────────────────────────────────────────────────────────
for d in ['data/raw', 'data/processed', 'results/raw',
          'results/aggregated', 'figures', 'checkpoints']:
    os.makedirs(d, exist_ok=True)

print('✅ Directories created.')
print(f'   Seeds: {SEEDS}')
print(f'   Domains: {DOMAINS}')

✅ Directories created.
   Seeds: [42, 123, 456]
   Domains: ['restaurant', 'laptop']


## Download & Parse SemEval-2014 Dataset

In [ ]:
import xml.etree.ElementTree as ET
from sklearn.model_selection import train_test_split

def parse_semeval_xml(filepath):
    tree = ET.parse(filepath)
    root = tree.getroot()
    records = []
    for sentence in root.iter('sentence'):
        text_el = sentence.find('text')
        if text_el is None or text_el.text is None:
            continue
        text = text_el.text.strip()
        aspect_terms = sentence.find('aspectTerms')
        if aspect_terms is None:
            continue
        for at in aspect_terms.findall('aspectTerm'):
            polarity = at.get('polarity')
            term     = at.get('term')
            if polarity == 'conflict' or not term:
                continue
            records.append({'sentence': text, 'aspect': term, 'sentiment': polarity})
    return pd.DataFrame(records)
dataset_root='/kaggle/input/datasets/charitarth/semeval-2014-task-4-aspectbasedsentimentanalysis'
SEMEVAL_FILES = {
    'restaurant_train': f'{dataset_root}/Restaurants_Train_v2.xml',
    'laptop_train':     f'{dataset_root}/Laptop_Train_v2.xml',
}

dfs = {}
for name, path in SEMEVAL_FILES.items():
    dfs[name] = parse_semeval_xml(path)

for domain in ['restaurant', 'laptop']:
    df = dfs[f'{domain}_train']
    train_df, test_df = train_test_split(
        df, test_size=0.2, random_state=42, stratify=df['sentiment']
    )
    train_df.to_csv(f'data/processed/{domain}_train.csv', index=False)
    test_df.to_csv(f'data/processed/{domain}_test.csv', index=False)
    dist = test_df['sentiment'].value_counts().to_dict()
    print(f'{domain}_train: {len(train_df)} | {domain}_test: {len(test_df)}')
    print(f'  test → pos={dist.get("positive",0)} neg={dist.get("negative",0)} neu={dist.get("neutral",0)}')

print('\n✅ All splits ready.')

restaurant_train: 2881 | restaurant_test: 721
  test → pos=433 neg=161 neu=127
laptop_train: 1850 | laptop_test: 463
  test → pos=198 neg=173 neu=92

✅ All splits ready.


In [ ]:
sample = pd.read_csv('data/processed/restaurant_train.csv')
print('Sample from Restaurant Train:')
sample[['sentence','aspect','sentiment']].head(5)

Sample from Restaurant Train:


,sentence,aspect,sentiment
0,Deliveries often take up to an hour and the pr...,prices,negative
1,I loved everythig about it-especially the show...,actors,positive
2,Truly the mark of an attentive waiter.,waiter,positive
3,"Great spot, whether looking for a couple of dr...",spot,positive
4,The lava cake dessert was incredible and I rec...,lava cake dessert,positive


## Energy Tracker Utility

In [ ]:
from codecarbon import EmissionsTracker

class ABSAEnergyTracker:
    def __init__(self, model_name, output_dir='results/raw'):
        self.model_name = model_name
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        self.tracker = EmissionsTracker(
            project_name=f'absa_{model_name}',
            output_dir=output_dir,
            log_level='error',
            save_to_file=True,
            measure_power_secs=5,
        )

    def start_training(self):
        tracemalloc.start()
        self._train_start = time.time()
        self.tracker.start()

    def stop_training(self):
        emissions_kg = self.tracker.stop()  
        self._train_time = time.time() - self._train_start
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        # CodeCarbon returns kg CO2eq; convert to kWh via UK grid intensity
        # UK grid: ~233 gCO2/kWh  →  kWh = kg_CO2 / 0.233
        energy_kwh = (emissions_kg / 0.233) if emissions_kg else 0.0
        return {
            'train_time_s':    round(self._train_time, 3),
            'energy_kwh':      round(energy_kwh, 8),
            'carbon_kg_co2':   round(emissions_kg or 0.0, 10),
            'peak_memory_mb':  round(peak / 1024 / 1024, 2),
        }

    def measure_inference_latency(self, predict_fn, X_test, n_runs=3):
        """Returns mean latency per sample (ms) across n_runs."""
        latencies = []
        for _ in range(n_runs):
            t0 = time.time()
            predict_fn(X_test)
            latencies.append((time.time() - t0) / len(X_test) * 1000)
        mean_lat = sum(latencies) / n_runs
        std_lat  = (sum((x - mean_lat)**2 for x in latencies) / n_runs) ** 0.5
        return {
            'latency_ms_per_sample': round(mean_lat, 4),
            'latency_std_ms':        round(std_lat, 4),
        }


def save_result(result_dict, filename):
    """Save experiment result as JSON."""
    path = f'results/raw/{filename}.json'
    with open(path, 'w') as f:
        json.dump(result_dict, f, indent=2)
    return path


print('✅ ABSAEnergyTracker ready.')

✅ ABSAEnergyTracker ready.


## Model 1: TF-IDF + LinearSVC (Classical Baseline)

In [6]:
# ── TF-IDF + LinearSVC ────────────────────────────────────────────────────────
# Based on: Schouten & Frasincar (2016) — Survey on Aspect-Level Sentiment Analysis
# Expected accuracy: ~74–78% Restaurant | ~68–74% Laptop
# This is the classical baseline. Very fast, very low energy.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

all_results = []   # Master results list — append to this every run

def run_tfidf_svm(domain, seed):
    np.random.seed(seed)
    train = pd.read_csv(f'data/processed/{domain}_train.csv').dropna()
    test  = pd.read_csv(f'data/processed/{domain}_test.csv').dropna()

    # Input: concatenate aspect + sentence (standard for classical ABSA)
    X_train = [f"{r.aspect} {r.sentence}" for _, r in train.iterrows()]
    X_test  = [f"{r.aspect} {r.sentence}" for _, r in test.iterrows()]
    y_train, y_test = train['sentiment'].tolist(), test['sentiment'].tolist()

    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(
            ngram_range=(1, 3),
            max_features=50000,
            sublinear_tf=True,
            min_df=2,
        )),
        ('clf', LinearSVC(C=0.5, max_iter=2000, random_state=seed)),
    ])

    tracker = ABSAEnergyTracker(f'tfidf_svm_{domain}_s{seed}')
    tracker.start_training()
    pipeline.fit(X_train, y_train)
    energy = tracker.stop_training()

    y_pred   = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    latency  = tracker.measure_inference_latency(pipeline.predict, X_test)

    result = {
        'model': 'TF-IDF+SVM', 'domain': domain, 'seed': seed,
        'accuracy': round(accuracy, 4), 'macro_f1': round(macro_f1, 4),
        'n_params': 'N/A',
        **energy, **latency,
    }
    save_result(result, f'tfidf_svm_{domain}_s{seed}')
    all_results.append(result)
    print(f'  [{domain:10s} seed={seed}]  Acc={accuracy:.4f}  F1={macro_f1:.4f}  '
          f'Energy={energy["energy_kwh"]:.2e} kWh  '
          f'Time={energy["train_time_s"]:.1f}s  '
          f'Lat={latency["latency_ms_per_sample"]:.3f}ms')
    return result


print('═'*70)
print('MODEL 1 — TF-IDF + LinearSVC')
print('═'*70)
for domain in DOMAINS:
    for seed in SEEDS:
        run_tfidf_svm(domain, seed)
print('\n✅ TF-IDF+SVM complete.')

══════════════════════════════════════════════════════════════════════
MODEL 1 — TF-IDF + LinearSVC
══════════════════════════════════════════════════════════════════════
  [restaurant seed=42]  Acc=0.6976  F1=0.5832  Energy=4.54e-05 kWh  Time=1.4s  Lat=0.053ms
  [restaurant seed=123]  Acc=0.6976  F1=0.5832  Energy=4.77e-05 kWh  Time=1.4s  Lat=0.054ms
  [restaurant seed=456]  Acc=0.6976  F1=0.5832  Energy=4.50e-05 kWh  Time=1.3s  Lat=0.053ms
  [laptop     seed=42]  Acc=0.7127  F1=0.6791  Energy=3.57e-05 kWh  Time=1.1s  Lat=0.059ms
  [laptop     seed=123]  Acc=0.7127  F1=0.6791  Energy=3.34e-05 kWh  Time=1.0s  Lat=0.058ms
  [laptop     seed=456]  Acc=0.7127  F1=0.6791  Energy=3.31e-05 kWh  Time=1.0s  Lat=0.053ms

✅ TF-IDF+SVM complete.


## Model 2: FastText

In [7]:
# ── FastText ──────────────────────────────────────────────────────────────────
# Lightweight neural word embedding model.
# Expected accuracy: ~80–83% Restaurant | ~72–78% Laptop
# ~200× less energy than DistilBERT — key finding for your paper.

import fasttext

def run_fasttext(domain, seed):
    np.random.seed(seed)
    train = pd.read_csv(f'data/processed/{domain}_train.csv').dropna()
    test  = pd.read_csv(f'data/processed/{domain}_test.csv').dropna()

    def to_fasttext_format(df):
        lines = []
        for _, r in df.iterrows():
            txt = f'{r.aspect} {r.sentence}'.replace('\n', ' ').strip()
            lines.append(f'__label__{r.sentiment} {txt}')
        return lines

    # Write temp training file (FastText requires a file path)
    with tempfile.NamedTemporaryFile(mode='w', suffix='.txt',
                                     delete=False, encoding='utf-8') as f:
        f.write('\n'.join(to_fasttext_format(train)))
        train_file = f.name

    test_texts = [f'{r.aspect} {r.sentence}' for _, r in test.iterrows()]
    y_true_raw = [f'__label__{s}' for s in test['sentiment']]

    tracker = ABSAEnergyTracker(f'fasttext_{domain}_s{seed}')
    tracker.start_training()
    model = fasttext.train_supervised(
        input=train_file, epoch=25, lr=0.5,
        wordNgrams=2, dim=100, loss='softmax', seed=seed, verbose=0,
    )
    energy = tracker.stop_training()
    os.unlink(train_file)

    def ft_predict(text):
        labels, _ = model.predict(text)
        return labels[0]

    y_pred_raw = [ft_predict(t) for t in test_texts]
    y_true = [s.replace('__label__', '') for s in y_true_raw]
    y_pred = [s.replace('__label__', '') for s in y_pred_raw]

    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    latency  = tracker.measure_inference_latency(
        lambda texts: [ft_predict(t) for t in texts], test_texts
    )

    result = {
        'model': 'FastText', 'domain': domain, 'seed': seed,
        'accuracy': round(accuracy, 4), 'macro_f1': round(macro_f1, 4),
        'n_params': '~2M',
        **energy, **latency,
    }
    save_result(result, f'fasttext_{domain}_s{seed}')
    all_results.append(result)
    print(f'  [{domain:10s} seed={seed}]  Acc={accuracy:.4f}  F1={macro_f1:.4f}  '
          f'Energy={energy["energy_kwh"]:.2e} kWh  '
          f'Time={energy["train_time_s"]:.1f}s  '
          f'Lat={latency["latency_ms_per_sample"]:.3f}ms')
    return result


print('═'*70)
print('MODEL 2 — FastText')
print('═'*70)
for domain in DOMAINS:
    for seed in SEEDS:
        run_fasttext(domain, seed)
print('\n✅ FastText complete.')
print(all_results)

══════════════════════════════════════════════════════════════════════
MODEL 2 — FastText
══════════════════════════════════════════════════════════════════════
  [restaurant seed=42]  Acc=0.6921  F1=0.5992  Energy=4.76e-05 kWh  Time=1.4s  Lat=0.018ms
  [restaurant seed=123]  Acc=0.6935  F1=0.5963  Energy=4.72e-05 kWh  Time=1.4s  Lat=0.021ms
  [restaurant seed=456]  Acc=0.6949  F1=0.5959  Energy=4.71e-05 kWh  Time=1.4s  Lat=0.021ms
  [laptop     seed=42]  Acc=0.6998  F1=0.6678  Energy=4.02e-05 kWh  Time=1.2s  Lat=0.022ms
  [laptop     seed=123]  Acc=0.6998  F1=0.6678  Energy=4.03e-05 kWh  Time=1.2s  Lat=0.023ms
  [laptop     seed=456]  Acc=0.7041  F1=0.6709  Energy=4.07e-05 kWh  Time=1.2s  Lat=0.029ms

✅ FastText complete.
[{'model': 'TF-IDF+SVM', 'domain': 'restaurant', 'seed': 42, 'accuracy': 0.6976, 'macro_f1': 0.5832, 'n_params': 'N/A', 'train_time_s': 1.358, 'energy_kwh': 4.54e-05, 'carbon_kg_co2': 1.05777e-05, 'peak_memory_mb': 12.46, 'latency_ms_per_sample': 0.053, 'latency_std_

In [8]:
# ── Fix fasttext source file directly ────────────────────────────────────────
import fasttext.FastText as _ft
import inspect, os

# Find the file
ft_file = inspect.getfile(_ft)
print(f'Patching: {ft_file}')

with open(ft_file, 'r') as f:
    source = f.read()

# Apply the fix
patched = source.replace(
    'return labels, np.array(probs, copy=False)',
    'return labels, np.asarray(probs)'
)

with open(ft_file, 'w') as f:
    f.write(patched)

print('✅ Source file patched.')

# Reload the module
import importlib
importlib.reload(_ft)
import fasttext
importlib.reload(fasttext)

print('✅ fasttext reloaded. Now re-run the FastText cell.')

Patching: /usr/local/lib/python3.12/dist-packages/fasttext/FastText.py
✅ Source file patched.
✅ fasttext reloaded. Now re-run the FastText cell.


## Model 3: DistilBERT (Efficient Transformer)

In [12]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install',
                'transformers==4.40.0',
                'accelerate==0.29.0',
                'peft==0.10.0',
                '-q'], check=True)

print('✅ Done. Restart session then re-run all cells from top.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.3/297.3 kB 7.5 MB/s eta 0:00:00
✅ Done. Restart session then re-run all cells from top.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.


In [13]:
all_results=[{'model': 'TF-IDF+SVM', 'domain': 'restaurant', 'seed': 42, 'accuracy': 0.6976, 'macro_f1': 0.5832, 'n_params': 'N/A', 'train_time_s': 1.358, 'energy_kwh': 4.54e-05, 'carbon_kg_co2': 1.05777e-05, 'peak_memory_mb': 12.46, 'latency_ms_per_sample': 0.053, 'latency_std_ms': 0.0003}, {'model': 'TF-IDF+SVM', 'domain': 'restaurant', 'seed': 123, 'accuracy': 0.6976, 'macro_f1': 0.5832, 'n_params': 'N/A', 'train_time_s': 1.426, 'energy_kwh': 4.77e-05, 'carbon_kg_co2': 1.11143e-05, 'peak_memory_mb': 12.34, 'latency_ms_per_sample': 0.0544, 'latency_std_ms': 0.0007}, {'model': 'TF-IDF+SVM', 'domain': 'restaurant', 'seed': 456, 'accuracy': 0.6976, 'macro_f1': 0.5832, 'n_params': 'N/A', 'train_time_s': 1.348, 'energy_kwh': 4.502e-05, 'carbon_kg_co2': 1.04885e-05, 'peak_memory_mb': 12.35, 'latency_ms_per_sample': 0.0529, 'latency_std_ms': 0.001}, {'model': 'TF-IDF+SVM', 'domain': 'laptop', 'seed': 42, 'accuracy': 0.7127, 'macro_f1': 0.6791, 'n_params': 'N/A', 'train_time_s': 1.079, 'energy_kwh': 3.571e-05, 'carbon_kg_co2': 8.3213e-06, 'peak_memory_mb': 9.17, 'latency_ms_per_sample': 0.0588, 'latency_std_ms': 0.0006}, {'model': 'TF-IDF+SVM', 'domain': 'laptop', 'seed': 123, 'accuracy': 0.7127, 'macro_f1': 0.6791, 'n_params': 'N/A', 'train_time_s': 1.016, 'energy_kwh': 3.343e-05, 'carbon_kg_co2': 7.7883e-06, 'peak_memory_mb': 9.17, 'latency_ms_per_sample': 0.058, 'latency_std_ms': 0.0011}, {'model': 'TF-IDF+SVM', 'domain': 'laptop', 'seed': 456, 'accuracy': 0.7127, 'macro_f1': 0.6791, 'n_params': 'N/A', 'train_time_s': 1.002, 'energy_kwh': 3.314e-05, 'carbon_kg_co2': 7.7211e-06, 'peak_memory_mb': 9.17, 'latency_ms_per_sample': 0.0531, 'latency_std_ms': 0.0005}, {'model': 'FastText', 'domain': 'restaurant', 'seed': 42, 'accuracy': 0.6921, 'macro_f1': 0.5992, 'n_params': '~2M', 'train_time_s': 1.424, 'energy_kwh': 4.758e-05, 'carbon_kg_co2': 1.10871e-05, 'peak_memory_mb': 0.34, 'latency_ms_per_sample': 0.0181, 'latency_std_ms': 0.0004}, {'model': 'FastText', 'domain': 'restaurant', 'seed': 123, 'accuracy': 0.6935, 'macro_f1': 0.5963, 'n_params': '~2M', 'train_time_s': 1.41, 'energy_kwh': 4.716e-05, 'carbon_kg_co2': 1.09872e-05, 'peak_memory_mb': 0.34, 'latency_ms_per_sample': 0.0213, 'latency_std_ms': 0.0008}, {'model': 'FastText', 'domain': 'restaurant', 'seed': 456, 'accuracy': 0.6949, 'macro_f1': 0.5959, 'n_params': '~2M', 'train_time_s': 1.408, 'energy_kwh': 4.71e-05, 'carbon_kg_co2': 1.09747e-05, 'peak_memory_mb': 0.33, 'latency_ms_per_sample': 0.0214, 'latency_std_ms': 0.0004}, {'model': 'FastText', 'domain': 'laptop', 'seed': 42, 'accuracy': 0.6998, 'macro_f1': 0.6678, 'n_params': '~2M', 'train_time_s': 1.208, 'energy_kwh': 4.017e-05, 'carbon_kg_co2': 9.3599e-06, 'peak_memory_mb': 0.34, 'latency_ms_per_sample': 0.0221, 'latency_std_ms': 0.0009}, {'model': 'FastText', 'domain': 'laptop', 'seed': 123, 'accuracy': 0.6998, 'macro_f1': 0.6678, 'n_params': '~2M', 'train_time_s': 1.214, 'energy_kwh': 4.032e-05, 'carbon_kg_co2': 9.3955e-06, 'peak_memory_mb': 0.33, 'latency_ms_per_sample': 0.023, 'latency_std_ms': 0.0005}, {'model': 'FastText', 'domain': 'laptop', 'seed': 456, 'accuracy': 0.7041, 'macro_f1': 0.6709, 'n_params': '~2M', 'train_time_s': 1.233, 'energy_kwh': 4.066e-05, 'carbon_kg_co2': 9.474e-06, 'peak_memory_mb': 0.33, 'latency_ms_per_sample': 0.0292, 'latency_std_ms': 0.0008}]

In [14]:
# ── DistilBERT ────────────────────────────────────────────────────────────────
# Sanh et al. (2019) — 40% smaller, 60% faster than BERT-base.
# Retains 97% of BERT performance on GLUE.
# Input format: [CLS] sentence [SEP] aspect [SEP]  (Sun et al. 2019 BERT-ABSA)
# Expected accuracy: ~84–87% Restaurant | ~77–82% Laptop
# CRITICAL: no_cuda=True forces CPU — this is your study's core condition.

import torch
from transformers import (
    DistilBertTokenizer, DistilBertForSequenceClassification,
    TrainingArguments, Trainer,
)
from datasets import Dataset as HFDataset

def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro'),
    }

def make_hf_dataset(df, tokenizer, max_length=128):
    df = df.dropna().copy()
    df['label'] = df['sentiment'].map(LABEL2ID)
    ds = HFDataset.from_pandas(df[['sentence', 'aspect', 'label']].reset_index(drop=True))
    def tokenize_fn(examples):
        return tokenizer(
            examples['sentence'], examples['aspect'],
            padding='max_length', truncation=True, max_length=max_length,
        )
    return ds.map(tokenize_fn, batched=True)


def run_distilbert(domain, seed):
    torch.manual_seed(seed); np.random.seed(seed)

    train_df = pd.read_csv(f'data/processed/{domain}_train.csv')
    test_df  = pd.read_csv(f'data/processed/{domain}_test.csv')

    tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
    train_tok = make_hf_dataset(train_df, tokenizer)
    test_tok  = make_hf_dataset(test_df,  tokenizer)

    model = DistilBertForSequenceClassification.from_pretrained(
        'distilbert-base-uncased',
        num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID,
    )

    args = TrainingArguments(
        output_dir=f'checkpoints/distilbert_{domain}_s{seed}',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='no',
        seed=seed,           # ← CRITICAL: Force CPU
        dataloader_num_workers=0,
        report_to='none',
        logging_steps=999999,   # Suppress per-step logs
    )

    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_tok, eval_dataset=test_tok,
        compute_metrics=compute_metrics,
    )

    tracker = ABSAEnergyTracker(f'distilbert_{domain}_s{seed}')
    tracker.start_training()
    trainer.train()
    energy = tracker.stop_training()

    eval_res = trainer.evaluate()

    t0 = time.time()
    for _ in range(3): trainer.predict(test_tok)
    lat_ms = (time.time() - t0) / 3 / len(test_df.dropna()) * 1000

    result = {
        'model': 'DistilBERT', 'domain': domain, 'seed': seed,
        'accuracy': round(eval_res['eval_accuracy'], 4),
        'macro_f1': round(eval_res['eval_macro_f1'], 4),
        'n_params': '66M',
        'latency_ms_per_sample': round(lat_ms, 4),
        **energy,
    }
    save_result(result, f'distilbert_{domain}_s{seed}')
    all_results.append(result)
    print(f'  [{domain:10s} seed={seed}]  Acc={result["accuracy"]:.4f}  '
          f'F1={result["macro_f1"]:.4f}  '
          f'Energy={energy["energy_kwh"]:.4f} kWh  '
          f'Time={energy["train_time_s"]:.0f}s')
    return result

print('═'*70)
print('MODEL 3 — DistilBERT  (⚠️ Slow on CPU — ~30-60 min per domain)')
print('═'*70)
for domain in DOMAINS:
    for seed in SEEDS:
        run_distilbert(domain, seed)
print('\n✅ DistilBERT complete.')

══════════════════════════════════════════════════════════════════════
MODEL 3 — DistilBERT  (⚠️ Slow on CPU — ~30-60 min per domain)
══════════════════════════════════════════════════════════════════════


Map:   0%|          | 0/2881 [00:00<?, ? examples/s]

Map:   0%|          | 0/721 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Map:   0%|          | 0/463 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.641600,0.771058,0.714648
2,No log,0.616657,0.771058,0.731193
3,No log,0.627303,0.760259,0.729750


  [laptop     seed=456]  Acc=0.7603  F1=0.7297  Energy=0.0770 kWh  Time=2243s

✅ DistilBERT complete.


## Model 4: ALBERT (Parameter-Sharing Transformer)

In [15]:
print(all_results)

[{'model': 'TF-IDF+SVM', 'domain': 'restaurant', 'seed': 42, 'accuracy': 0.6976, 'macro_f1': 0.5832, 'n_params': 'N/A', 'train_time_s': 1.358, 'energy_kwh': 4.54e-05, 'carbon_kg_co2': 1.05777e-05, 'peak_memory_mb': 12.46, 'latency_ms_per_sample': 0.053, 'latency_std_ms': 0.0003}, {'model': 'TF-IDF+SVM', 'domain': 'restaurant', 'seed': 123, 'accuracy': 0.6976, 'macro_f1': 0.5832, 'n_params': 'N/A', 'train_time_s': 1.426, 'energy_kwh': 4.77e-05, 'carbon_kg_co2': 1.11143e-05, 'peak_memory_mb': 12.34, 'latency_ms_per_sample': 0.0544, 'latency_std_ms': 0.0007}, {'model': 'TF-IDF+SVM', 'domain': 'restaurant', 'seed': 456, 'accuracy': 0.6976, 'macro_f1': 0.5832, 'n_params': 'N/A', 'train_time_s': 1.348, 'energy_kwh': 4.502e-05, 'carbon_kg_co2': 1.04885e-05, 'peak_memory_mb': 12.35, 'latency_ms_per_sample': 0.0529, 'latency_std_ms': 0.001}, {'model': 'TF-IDF+SVM', 'domain': 'laptop', 'seed': 42, 'accuracy': 0.7127, 'macro_f1': 0.6791, 'n_params': 'N/A', 'train_time_s': 1.079, 'energy_kwh': 3.5

In [9]:
import torch
import numpy as np
import pandas as pd
import time
all_results=[{'model': 'TF-IDF+SVM', 'domain': 'restaurant', 'seed': 42, 'accuracy': 0.6976, 'macro_f1': 0.5832, 'n_params': 'N/A', 'train_time_s': 1.358, 'energy_kwh': 4.54e-05, 'carbon_kg_co2': 1.05777e-05, 'peak_memory_mb': 12.46, 'latency_ms_per_sample': 0.053, 'latency_std_ms': 0.0003}, {'model': 'TF-IDF+SVM', 'domain': 'restaurant', 'seed': 123, 'accuracy': 0.6976, 'macro_f1': 0.5832, 'n_params': 'N/A', 'train_time_s': 1.426, 'energy_kwh': 4.77e-05, 'carbon_kg_co2': 1.11143e-05, 'peak_memory_mb': 12.34, 'latency_ms_per_sample': 0.0544, 'latency_std_ms': 0.0007}, {'model': 'TF-IDF+SVM', 'domain': 'restaurant', 'seed': 456, 'accuracy': 0.6976, 'macro_f1': 0.5832, 'n_params': 'N/A', 'train_time_s': 1.348, 'energy_kwh': 4.502e-05, 'carbon_kg_co2': 1.04885e-05, 'peak_memory_mb': 12.35, 'latency_ms_per_sample': 0.0529, 'latency_std_ms': 0.001}, {'model': 'TF-IDF+SVM', 'domain': 'laptop', 'seed': 42, 'accuracy': 0.7127, 'macro_f1': 0.6791, 'n_params': 'N/A', 'train_time_s': 1.079, 'energy_kwh': 3.571e-05, 'carbon_kg_co2': 8.3213e-06, 'peak_memory_mb': 9.17, 'latency_ms_per_sample': 0.0588, 'latency_std_ms': 0.0006}, {'model': 'TF-IDF+SVM', 'domain': 'laptop', 'seed': 123, 'accuracy': 0.7127, 'macro_f1': 0.6791, 'n_params': 'N/A', 'train_time_s': 1.016, 'energy_kwh': 3.343e-05, 'carbon_kg_co2': 7.7883e-06, 'peak_memory_mb': 9.17, 'latency_ms_per_sample': 0.058, 'latency_std_ms': 0.0011}, {'model': 'TF-IDF+SVM', 'domain': 'laptop', 'seed': 456, 'accuracy': 0.7127, 'macro_f1': 0.6791, 'n_params': 'N/A', 'train_time_s': 1.002, 'energy_kwh': 3.314e-05, 'carbon_kg_co2': 7.7211e-06, 'peak_memory_mb': 9.17, 'latency_ms_per_sample': 0.0531, 'latency_std_ms': 0.0005}, {'model': 'FastText', 'domain': 'restaurant', 'seed': 42, 'accuracy': 0.6921, 'macro_f1': 0.5992, 'n_params': '~2M', 'train_time_s': 1.424, 'energy_kwh': 4.758e-05, 'carbon_kg_co2': 1.10871e-05, 'peak_memory_mb': 0.34, 'latency_ms_per_sample': 0.0181, 'latency_std_ms': 0.0004}, {'model': 'FastText', 'domain': 'restaurant', 'seed': 123, 'accuracy': 0.6935, 'macro_f1': 0.5963, 'n_params': '~2M', 'train_time_s': 1.41, 'energy_kwh': 4.716e-05, 'carbon_kg_co2': 1.09872e-05, 'peak_memory_mb': 0.34, 'latency_ms_per_sample': 0.0213, 'latency_std_ms': 0.0008}, {'model': 'FastText', 'domain': 'restaurant', 'seed': 456, 'accuracy': 0.6949, 'macro_f1': 0.5959, 'n_params': '~2M', 'train_time_s': 1.408, 'energy_kwh': 4.71e-05, 'carbon_kg_co2': 1.09747e-05, 'peak_memory_mb': 0.33, 'latency_ms_per_sample': 0.0214, 'latency_std_ms': 0.0004}, {'model': 'FastText', 'domain': 'laptop', 'seed': 42, 'accuracy': 0.6998, 'macro_f1': 0.6678, 'n_params': '~2M', 'train_time_s': 1.208, 'energy_kwh': 4.017e-05, 'carbon_kg_co2': 9.3599e-06, 'peak_memory_mb': 0.34, 'latency_ms_per_sample': 0.0221, 'latency_std_ms': 0.0009}, {'model': 'FastText', 'domain': 'laptop', 'seed': 123, 'accuracy': 0.6998, 'macro_f1': 0.6678, 'n_params': '~2M', 'train_time_s': 1.214, 'energy_kwh': 4.032e-05, 'carbon_kg_co2': 9.3955e-06, 'peak_memory_mb': 0.33, 'latency_ms_per_sample': 0.023, 'latency_std_ms': 0.0005}, {'model': 'FastText', 'domain': 'laptop', 'seed': 456, 'accuracy': 0.7041, 'macro_f1': 0.6709, 'n_params': '~2M', 'train_time_s': 1.233, 'energy_kwh': 4.066e-05, 'carbon_kg_co2': 9.474e-06, 'peak_memory_mb': 0.33, 'latency_ms_per_sample': 0.0292, 'latency_std_ms': 0.0008}, {'model': 'DistilBERT', 'domain': 'restaurant', 'seed': 42, 'accuracy': 0.7684, 'macro_f1': 0.6869, 'n_params': '66M', 'latency_ms_per_sample': 109.9489, 'train_time_s': 3431.163, 'energy_kwh': 0.11788547, 'carbon_kg_co2': 0.0274673139, 'peak_memory_mb': 0.68}, {'model': 'DistilBERT', 'domain': 'restaurant', 'seed': 123, 'accuracy': 0.7656, 'macro_f1': 0.6686, 'n_params': '66M', 'latency_ms_per_sample': 108.3417, 'train_time_s': 3492.474, 'energy_kwh': 0.11999204, 'carbon_kg_co2': 0.0279581453, 'peak_memory_mb': 0.67}, {'model': 'DistilBERT', 'domain': 'restaurant', 'seed': 456, 'accuracy': 0.7739, 'macro_f1': 0.68, 'n_params': '66M', 'latency_ms_per_sample': 109.5446, 'train_time_s': 3520.029, 'energy_kwh': 0.12093914, 'carbon_kg_co2': 0.0281788199, 'peak_memory_mb': 0.67}, {'model': 'DistilBERT', 'domain': 'laptop', 'seed': 42, 'accuracy': 0.7689, 'macro_f1': 0.7374, 'n_params': '66M', 'latency_ms_per_sample': 109.064, 'train_time_s': 2270.449, 'energy_kwh': 0.0780055, 'carbon_kg_co2': 0.0181752822, 'peak_memory_mb': 0.62}, {'model': 'DistilBERT', 'domain': 'laptop', 'seed': 123, 'accuracy': 0.7732, 'macro_f1': 0.7417, 'n_params': '66M', 'latency_ms_per_sample': 108.68, 'train_time_s': 2246.354, 'energy_kwh': 0.07717825, 'carbon_kg_co2': 0.0179825333, 'peak_memory_mb': 0.63}, {'model': 'DistilBERT', 'domain': 'laptop', 'seed': 456, 'accuracy': 0.7603, 'macro_f1': 0.7297, 'n_params': '66M', 'latency_ms_per_sample': 109.7281, 'train_time_s': 2242.542, 'energy_kwh': 0.07704743, 'carbon_kg_co2': 0.0179520514, 'peak_memory_mb': 0.63}]

In [17]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install',
                'transformers==4.40.0',
                'accelerate==0.29.0',
                'peft==0.10.0',
                '--force-reinstall',
                '-q'], check=True)

print('✅ Done. YOU MUST NOW: Kaggle top menu → Run → Restart & Clear Outputs')
print('Then re-run ALL cells from Section 0 downward.')
print('Do NOT skip the restart — the old broken peft is still loaded in memory.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.3/297.3 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.0/802.0 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 2.16.1 requires fsspec[http]<=2023.10.0,>=2023.1.0, but you have fsspec 2026.2.0 which is incompatible.
ydata-profiling 4.18.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.3 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
opencv-contrib-python 4.12.0.88 requires n

✅ Done. YOU MUST NOW: Kaggle top menu → Run → Restart & Clear Outputs
Then re-run ALL cells from Section 0 downward.
Do NOT skip the restart — the old broken peft is still loaded in memory.


In [29]:
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'transformers==4.35.2',
                'accelerate==0.24.1',
                '--force-reinstall',
                '-q'], check=True)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.5/123.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.4/261.4 kB 12.9 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 2.16.1 requires fsspec[http]<=2023.10.0,>=2023.1.0, but you have fsspec 2026.2.0 which is incompatible.
ydata-profiling 4.18.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.3 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
opencv-contrib-python 4.12.0.88 requires n

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', 'transformers==4.35.2', 'accelerate==0.24.1', '--force-reinstall', '-q'], returncode=0)

In [10]:
# ── ALBERT ────────────────────────────────────────────────────────────────────
from datasets import Dataset as HFDataset
from transformers import (
    AlbertTokenizer, AlbertForSequenceClassification,
    TrainingArguments, Trainer,
)

def make_hf_dataset(df, tokenizer, max_length=128):
    df = df.dropna().copy()
    df['label'] = df['sentiment'].map(LABEL2ID)
    ds = HFDataset.from_pandas(df[['sentence', 'aspect', 'label']].reset_index(drop=True))
    def tokenize_fn(examples):
        return tokenizer(
            examples['sentence'], examples['aspect'],
            padding='max_length', truncation=True, max_length=max_length,
        )
    return ds.map(tokenize_fn, batched=True)

def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro'),
    }

def run_albert(domain, seed):
    torch.manual_seed(seed); np.random.seed(seed)

    train_df = pd.read_csv(f'data/processed/{domain}_train.csv')
    test_df  = pd.read_csv(f'data/processed/{domain}_test.csv')

    tokenizer = AlbertTokenizer.from_pretrained('albert-base-v2')
    train_tok = make_hf_dataset(train_df, tokenizer)
    test_tok  = make_hf_dataset(test_df,  tokenizer)

    model = AlbertForSequenceClassification.from_pretrained(
        'albert-base-v2',
        num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID,
    )

    args = TrainingArguments(
        output_dir=f'checkpoints/albert_{domain}_s{seed}',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='no',
        seed=seed,
        dataloader_num_workers=0,
        report_to='none',
        logging_steps=999999,
    )

    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_tok, eval_dataset=test_tok,
        compute_metrics=compute_metrics,
    )

    tracker = ABSAEnergyTracker(f'albert_{domain}_s{seed}')
    tracker.start_training()
    trainer.train()
    energy = tracker.stop_training()

    eval_res = trainer.evaluate()

    t0 = time.time()
    for _ in range(3): trainer.predict(test_tok)
    lat_ms = (time.time() - t0) / 3 / len(test_df.dropna()) * 1000

    result = {
        'model': 'ALBERT', 'domain': domain, 'seed': seed,
        'accuracy': round(eval_res['eval_accuracy'], 4),
        'macro_f1': round(eval_res['eval_macro_f1'], 4),
        'n_params': '12M',
        'latency_ms_per_sample': round(lat_ms, 4),
        **energy,
    }
    save_result(result, f'albert_{domain}_s{seed}')
    all_results.append(result)
    print(f'  [{domain:10s} seed={seed}]  Acc={result["accuracy"]:.4f}  '
          f'F1={result["macro_f1"]:.4f}  '
          f'Energy={energy["energy_kwh"]:.4f} kWh  '
          f'Time={energy["train_time_s"]:.0f}s')
    return result


print('═'*70)
print('MODEL 4 — ALBERT  (⚠️ Slow on CPU — ~40-70 min per domain)')
print('═'*70)
for domain in DOMAINS:
    for seed in SEEDS:
        run_albert(domain, seed)
print('\n✅ ALBERT complete.')

══════════════════════════════════════════════════════════════════════
MODEL 4 — ALBERT  (⚠️ Slow on CPU — ~40-70 min per domain)
══════════════════════════════════════════════════════════════════════


Map:   0%|          | 0/2881 [00:00<?, ? examples/s]

Map:   0%|          | 0/721 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/47.4M [00:00<?, ?B/s]

...

Map:   0%|          | 0/463 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertForSequenceClassification LOAD REPORT from: albert-base-v2
Key                          | Status     | 
-----------------------------+------------+-
predictions.LayerNorm.bias   | UNEXPECTED | 
predictions.decoder.bias     | UNEXPECTED | 
predictions.dense.weight     | UNEXPECTED | 
predictions.dense.bias       | UNEXPECTED | 
predictions.bias             | UNEXPECTED | 
predictions.LayerNorm.weight | UNEXPECTED | 
classifier.bias              | MISSING    | 
classifier.weight            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.654747,0.736501,0.637552
2,No log,0.576386,0.796976,0.734115
3,No log,0.534963,0.805616,0.772564


  [laptop     seed=456]  Acc=0.8056  F1=0.7726  Energy=0.2634 kWh  Time=2926s

✅ ALBERT complete.


## Model 5: MiniLM (Lightweight Transformer)

In [12]:
# ── MiniLM ────────────────────────────────────────────────────────────────────
# Wang et al. (2020) — Deep Self-Attention Distillation.
# 2.7x faster than BERT-base, 50% fewer parameters.
# Expected to be Pareto-optimal: near-ALBERT accuracy at lower energy.
# Expected accuracy: ~84–87% Restaurant | ~77–82% Laptop

from transformers import AutoTokenizer, AutoModelForSequenceClassification

MINILM_MODEL = 'microsoft/MiniLM-L12-H384-uncased'

def run_minilm(domain, seed):
    torch.manual_seed(seed); np.random.seed(seed)

    train_df = pd.read_csv(f'data/processed/{domain}_train.csv')
    test_df  = pd.read_csv(f'data/processed/{domain}_test.csv')

    tokenizer = AutoTokenizer.from_pretrained(MINILM_MODEL)
    train_tok = make_hf_dataset(train_df, tokenizer)
    test_tok  = make_hf_dataset(test_df,  tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        MINILM_MODEL,
        num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )

    args = TrainingArguments(
        output_dir=f'checkpoints/minilm_{domain}_s{seed}',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=3e-5,     # Slightly higher LR — MiniLM benefits from this
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='no',
        seed=seed,
        dataloader_num_workers=0,
        report_to='none',
        logging_steps=999999,
    )

    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_tok, eval_dataset=test_tok,
        compute_metrics=compute_metrics,
    )

    tracker = ABSAEnergyTracker(f'minilm_{domain}_s{seed}')
    tracker.start_training()
    trainer.train()
    energy = tracker.stop_training()

    eval_res = trainer.evaluate()

    t0 = time.time()
    for _ in range(3): trainer.predict(test_tok)
    lat_ms = (time.time() - t0) / 3 / len(test_df.dropna()) * 1000

    result = {
        'model': 'MiniLM', 'domain': domain, 'seed': seed,
        'accuracy': round(eval_res['eval_accuracy'], 4),
        'macro_f1': round(eval_res['eval_macro_f1'], 4),
        'n_params': '33M',
        'latency_ms_per_sample': round(lat_ms, 4),
        **energy,
    }
    save_result(result, f'minilm_{domain}_s{seed}')
    all_results.append(result)
    print(f'  [{domain:10s} seed={seed}]  Acc={result["accuracy"]:.4f}  '
          f'F1={result["macro_f1"]:.4f}  '
          f'Energy={energy["energy_kwh"]:.4f} kWh  '
          f'Time={energy["train_time_s"]:.0f}s')
    return result


print('═'*70)
print('MODEL 5 — MiniLM  (⚠️ Slow on CPU — ~25-50 min per domain)')
print('═'*70)
for domain in DOMAINS:
    for seed in SEEDS:
        run_minilm(domain, seed)
print('\n✅ MiniLM complete.')

══════════════════════════════════════════════════════════════════════
MODEL 5 — MiniLM  (⚠️ Slow on CPU — ~25-50 min per domain)
══════════════════════════════════════════════════════════════════════


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Map:   0%|          | 0/463 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: microsoft/MiniLM-L12-H384-uncased
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.844965,0.665227,0.488512
2,No log,0.706142,0.758099,0.694474
3,No log,0.675042,0.773218,0.724003


  [laptop     seed=456]  Acc=0.7732  F1=0.7240  Energy=0.0820 kWh  Time=911s

✅ MiniLM complete.
